# Part 8 · ECoRe innovations 1–3

This notebook freezes the Part 7 encoder and tests three claims on source-heldout evidence: source-balanced author distributions, environment-relative geometry, and author-heldout episodic transfer. It does not rebuild Part 7 embeddings and does not train a production encoder.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, shutil, subprocess, sys
import pandas as pd

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/style_matching')
os.chdir(REPO)
EXP = REPO / 'artifacts/source_expansion_v2'
HELDOUT = REPO / 'data/all/meta/all_source_heldout_splits.parquet'
EMBEDDINGS = EXP / 'expanded_source_heldout_eval'
OUT = EXP / 'ecore_innovations_v1'
assert HELDOUT.exists(), f'Run Parts 1–7 first; missing {HELDOUT}'
for name in ('style_embedding_train_embeddings.npy', 'style_embedding_eval_embeddings.npy', 'style_embedding_scores.npz'):
    assert (EMBEDDINGS / name).exists(), f'Part 7 embedding artifact missing: {EMBEDDINGS / name}'
assert (REPO / 'scripts/evaluate_ecore_innovations.py').exists(), 'Pull the commit containing the redesigned Part 8 script'

def run(cmd):
    print('>>>', ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'command failed with exit {code}: {" ".join(map(str, cmd))}')

# Remove only interrupted legacy Part 8 model directories. Completed checkpoints are preserved.
legacy_root = EXP / 'top4_full_retrain'
if legacy_root.exists():
    for candidate in legacy_root.glob('*/finetuned_authorship_expanded'):
        if not (candidate / 'training_config.json').exists():
            print('REMOVE interrupted legacy checkpoint:', candidate)
            shutil.rmtree(candidate)
    for model_name in ('strict_eval_finetuned_authorship', 'production_full_source_finetuned_authorship'):
        for candidate in legacy_root.glob(f'*/{model_name}'):
            if not (candidate / 'training_config.json').exists():
                print('REMOVE interrupted checkpoint:', candidate)
                shutil.rmtree(candidate)

print('Frozen input:', HELDOUT)
print('Reusing Part 7 embeddings:', EMBEDDINGS)
print('No encoder batches or production retraining will run in Part 8.')

## 8.1–8.3 · Falsifiable tests

Innovation 1 compares one centroid, hard prototype, and source-balanced soft prototypes. Innovation 2 uses language × register cohort-centred residual geometry and a shuffled-environment control. Innovation 3 trains a candidate-identity-free pairwise scorer on variable support episodes and evaluates it by whole-author cross-fitting.

In [ ]:
run([
    sys.executable, 'scripts/evaluate_ecore_innovations.py',
    '--input', str(HELDOUT),
    '--embedding-dir', str(EMBEDDINGS),
    '--output-dir', str(OUT),
    '--temperature', '0.08',
    '--author-folds', '5',
    '--hard-negatives', '12',
    '--bootstrap-runs', '5000',
    '--train-cap', '300',
    '--embedding-seed', '20260701',
    '--seed', '20260725',
])

report_path = OUT / 'ecore_innovation_metrics.json'
report = json.loads(report_path.read_text())
summary = pd.DataFrame([
    {
        'innovation': 1,
        'claim': 'author as a source-balanced distribution',
        **report['innovation_1_distributional_profile']['paired_profile_bootstrap'],
        'supported': report['innovation_1_distributional_profile']['supported'],
    },
    {
        'innovation': 2,
        'claim': 'environment-relative evidence',
        **report['innovation_2_cohort_relative_geometry']['paired_profile_bootstrap'],
        'supported': report['innovation_2_cohort_relative_geometry']['supported'],
    },
    {
        'innovation': 3,
        'claim': 'whole-author episodic transfer',
        **report['innovation_3_episodic_transfer']['paired_profile_bootstrap'],
        'supported': report['innovation_3_episodic_transfer']['supported'],
    },
])
display(summary)
display(pd.DataFrame(report['test_metrics']).T.sort_values('mrr', ascending=False))
display(pd.DataFrame(report['innovation_3_episodic_transfer']['variable_support_test_metrics']).T)
print('RETURN:', report_path)
print('RETURN:', OUT / 'ecore_innovation_scores.npz')

## 8.4 · Gutenberg-eligible author expansion

This stage verifies a research-curated list of high-yield modern authors through targeted Gutendex queries; it does not scan the catalog. Admission still requires at least three independently titled, single-author, original-language, public-domain plaintext works. Existing registry rows are preserved; the public Author Library is filtered later from the completed index.

In [ ]:
GUTENBERG = EXP / 'gutenberg_targeted_v1'
GUTENBERG.mkdir(parents=True, exist_ok=True)
TARGETS = REPO / 'data/source_registry/gutenberg_target_authors_2026_07.csv'
CANDIDATES = GUTENBERG / 'verified_eligible_authors.csv'
MAX_NEW_AUTHORS_PER_LANGUAGE = 0  # 0 = all qualifying authors; reruns resume downloaded sources

assert TARGETS.exists(), f'Pull the targeted Gutenberg author list: {TARGETS}'
if CANDIDATES.exists():
    print('REUSE completed targeted Gutenberg verification:', CANDIDATES)
else:
    run([sys.executable, 'scripts/discover_listed_gutenberg_authors.py',
         '--candidates', str(TARGETS), '--output', str(CANDIDATES),
         '--min-works', '3', '--max-pages-per-author', '8', '--workers', '6'])
eligible = pd.read_csv(CANDIDATES)
assert 50 <= len(eligible) <= 100, f'Expected 50–100 verified authors, got {len(eligible)}'
LANGUAGES = tuple(sorted(eligible['original_language'].unique()))
display(eligible.groupby('original_language').size().rename('verified_authors'))
for language in LANGUAGES:
    run([sys.executable, 'scripts/fetch_gutendex.py', '--corpus', 'literary',
         '--language', language, '--registry', str(CANDIDATES),
         '--min-works', '3', '--max-works', '6',
         '--max-authors', str(MAX_NEW_AUTHORS_PER_LANGUAGE), '--skip-covered'])
print('RETURN:', CANDIDATES)
print('RETURN:', CANDIDATES.with_suffix('.report.json'))

## 8.5 · Rebuild frozen evidence once

The encoder remains frozen. New texts receive embeddings; previously cached chunks are reused. Source-heldout splits are rebuilt because the candidate universe changed.

In [ ]:
CHUNKS_V3 = REPO / 'data/all/meta/all_sources_chunks.parquet'
COVERAGE_V3 = GUTENBERG / 'coverage.json'
HELDOUT_V3 = GUTENBERG / 'source_heldout_splits.parquet'
HELDOUT_REPORT_V3 = GUTENBERG / 'source_heldout_report.json'
EVAL_V3 = GUTENBERG / 'frozen_encoder_eval'
INDEX_V3 = GUTENBERG / 'index'
MODEL = REPO / 'artifacts/multilingual_author_style_v1'

run([sys.executable, 'scripts/build_chunk_parquet_from_sources.py', '--corpus', 'both',
     '--output', str(CHUNKS_V3), '--coverage-output', str(COVERAGE_V3),
     '--min-sources', '3', '--min-chunks', '30'])
run([sys.executable, 'scripts/make_source_heldout_splits.py', '--input', str(CHUNKS_V3),
     '--output', str(HELDOUT_V3), '--report', str(HELDOUT_REPORT_V3)])
run([sys.executable, 'scripts/style_embedding_recall.py', '--input', str(HELDOUT_V3),
     '--out-dir', str(EVAL_V3), '--model-name', str(MODEL), '--batch-size', '128',
     '--train-cap', '300', '--eval-splits', 'dev,test', '--device', 'cuda',
     '--reuse-input', str(HELDOUT), '--reuse-embedding-dir', str(EMBEDDINGS), '--skip-existing'])
run([sys.executable, 'scripts/multilingual_style_index.py', 'build', '--input', str(CHUNKS_V3),
     '--out-dir', str(INDEX_V3), '--model-name', str(MODEL),
     '--topic-model-name', 'intfloat/multilingual-e5-base',
     '--embedding-cache', str(INDEX_V3 / 'chunk_embeddings.npz'),
     '--topic-embedding-cache', str(INDEX_V3 / 'topic_chunk_embeddings.npz'),
     '--batch-size', '128', '--per-source-cap', '50', '--profile-cap', '600',
     '--profile-strategy', 'single_centroid', '--heldout-report', str(HELDOUT_REPORT_V3),
     '--model-label', 'challenger_finetuned', '--artifact-version', 'gutenberg_all_v1',
     '--device', 'cuda'])

## 8.6 · Hubness punishment

Frequency and local-density corrections are tuned by source-grouped cross-fitting inside dev. Candidate penalties are standardized within language and source-support tier. Test is opened once; a correction is adopted only when its paired profile-bootstrap MRR interval is wholly positive and Recall@3 does not decrease.

In [ ]:
HUBNESS = GUTENBERG / 'hubness_correction'
run([sys.executable, 'scripts/evaluate_hubness_correction.py', '--input', str(HELDOUT_V3),
     '--scores', str(EVAL_V3 / 'style_embedding_scores.npz'), '--output-dir', str(HUBNESS),
     '--top-k', '10', '--folds', '5',
     '--lambdas', '0,0.02,0.05,0.08,0.1,0.15,0.2',
     '--bootstrap-runs', '5000', '--seed', '20260726'])
hub_report = json.loads((HUBNESS / 'hubness_correction_metrics.json').read_text())
display(pd.DataFrame({name: result['test'] for name, result in hub_report['methods'].items()}).T)
print('selected:', hub_report['selected'])
run([sys.executable, 'scripts/attach_hubness_correction.py', '--index-dir', str(INDEX_V3),
     '--report', str(HUBNESS / 'hubness_correction_metrics.json')])
print('RETURN:', HUBNESS / 'hubness_correction_metrics.json')

## 8.7 · Synchronize the public Author Library

Reviewed metadata for the 88 indexed additions is appended idempotently before export. Existing registry rows are never overwritten or destroyed; only authors present in the completed index enter the matchable library export.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
REPO = Path('/content/drive/MyDrive/style_matching')
os.chdir(REPO)
GUTENBERG = REPO / 'artifacts/source_expansion_v2/gutenberg_targeted_v1'
INDEX_V3 = GUTENBERG / 'index'
def run(cmd):
    print('>>>', ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'command failed with exit {code}: {" ".join(map(str, cmd))}')
assert (INDEX_V3 / 'profiles.parquet').exists(), f'Missing completed index: {INDEX_V3}'
LIBRARY_AUDIT = GUTENBERG / 'author_library_coverage.json'
ADDITIONS = REPO / 'data/source_registry/gutenberg_indexed_metadata_2026_07.csv'
assert ADDITIONS.exists(), f'Pull the reviewed metadata additions: {ADDITIONS}'
run([sys.executable, 'scripts/merge_author_registry_metadata.py',
     '--additions', str(ADDITIONS)])
run([sys.executable, 'scripts/export_author_library.py',
     '--profiles', str(INDEX_V3 / 'profiles.parquet'),
     '--coverage-output', str(LIBRARY_AUDIT)])
library_audit = json.loads(LIBRARY_AUDIT.read_text())
assert not library_audit['indexed_missing_registry_metadata'], library_audit['indexed_missing_registry_metadata']
assert library_audit['exported_authors'] == library_audit['indexed_authors']
print(json.dumps({key: value if not isinstance(value, list) else len(value)
                  for key, value in library_audit.items()}, indent=2))
print('NEW INDEX:', INDEX_V3)
print('RETURN:', LIBRARY_AUDIT)

## 8.8 · Compare old and new indexes on identical queries

Coverage and ranking are evaluated separately. The comparison reuses the frozen Part 8.5 test embeddings and reports: all expanded queries with missing old profiles scored as failures; shared-author queries against each operational candidate universe; and a shared-candidate control that isolates profile changes. No GPU encoding is repeated.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
REPO = Path('/content/drive/MyDrive/style_matching')
os.chdir(REPO)
GUTENBERG = REPO / 'artifacts/source_expansion_v2/gutenberg_targeted_v1'
HELDOUT_V3 = GUTENBERG / 'source_heldout_splits.parquet'
EVAL_V3 = GUTENBERG / 'frozen_encoder_eval'
INDEX_V3 = GUTENBERG / 'index'
def run(cmd):
    print('>>>', ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'command failed with exit {code}: {" ".join(map(str, cmd))}')
OLD_INDEX = REPO / 'artifacts/multilingual_style_index_challenger_v1'
COMPARISON = GUTENBERG / 'old_vs_new_index_metrics.json'
assert OLD_INDEX.exists(), f'Missing old production index: {OLD_INDEX}'
run([sys.executable, 'scripts/compare_index_retrieval.py',
     '--input', str(HELDOUT_V3),
     '--eval-embeddings', str(EVAL_V3 / 'style_embedding_eval_embeddings.npy'),
     '--eval-chunk-ids', str(EVAL_V3 / 'style_embedding_eval_chunk_ids.npy'),
     '--old-index', str(OLD_INDEX), '--new-index', str(INDEX_V3),
     '--output', str(COMPARISON), '--bootstrap-runs', '5000'])
comparison = json.loads(COMPARISON.read_text())
display(pd.DataFrame({
    'expanded coverage-adjusted': comparison['expanded_coverage_adjusted'],
    'shared authors / operational candidates': comparison['shared_true_full_candidate_universe'],
    'shared authors / shared candidates': comparison['shared_true_shared_candidate_control'],
}).T[['n_queries', 'n_true_profiles', 'old', 'new', 'mrr_delta', 'recall_at_1_delta', 'recall_at_3_delta', 'recall_at_5_delta', 'recall_at_20_delta']])
print('NEW INDEX BETTER:', comparison['new_index_better'])
print('RETURN:', COMPARISON)

## Decision rule

A claim passes only when its paired author-profile bootstrap 95% interval is wholly above zero. Innovation 2 must additionally beat the shuffled-environment control. Unsupported claims remain negative results; Part 8 does not hide them by training a larger encoder or adding more indicators.